# Load a fine-tuned run from the Hub

Downloads one experiment's folder from the shared `hub.repo_id` model repo (as written by `src/hub_sync.py` during training -- see `README.md`), loads the base model + LoRA adapter, and runs it on a sample Whisper transcript.

Set `HUB_REPO_ID` and `RUN_FOLDER` below to whichever experiment you want (e.g. the folder names printed in `run_sweep.py`'s summary table, or the `output_dir` basename of a single `train.py` run).

In [ ]:
%pip install -q transformers peft torch huggingface_hub pyyaml python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # picks up HF_TOKEN from a .env in the repo root, if you're not already `huggingface-cli login`'d

In [ ]:
HUB_REPO_ID = "PedramR/ASR_Post-processing"  # the shared repo every run's config.yaml points at
RUN_FOLDER = "qwen3.5-2b"  # this experiment's folder within that repo
# None -> the final (last-step) saved adapter at the folder root
# "best_checkpoint_wer" -> the best-by-test-WER checkpoint (see README's "Best checkpoint by WER")
# "checkpoint-800" (etc.) -> a specific numbered checkpoint
CHECKPOINT = "best_checkpoint_wer"
LOCAL_DIR = "./downloaded_model"

## Download

`allow_patterns` limits the download to just this run's folder -- other experiments in the same shared repo are left untouched.

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=HUB_REPO_ID,
    allow_patterns=[f"{RUN_FOLDER}/**"],
    local_dir=LOCAL_DIR,
)

run_dir = Path(LOCAL_DIR) / RUN_FOLDER
adapter_dir = run_dir / CHECKPOINT if CHECKPOINT else run_dir
print("run_dir:", run_dir)
print("adapter_dir:", adapter_dir)
print("contents:", sorted(p.name for p in run_dir.iterdir()))

## Read this run's config

`train.py` writes `config.yaml` into `output_dir` at the start of every run, so it's part of what got synced here -- this is how we know which base model and system prompt to use without hardcoding them.

In [ ]:
import yaml

run_config = yaml.safe_load((run_dir / "config.yaml").read_text())
MODEL_ID = run_config["model_id"]
SYSTEM_PROMPT = run_config["system_prompt"]
print("model_id:", MODEL_ID)
print("system_prompt:", SYSTEM_PROMPT)

`system_prompt` is `null` when the run used `data.py`'s default -- fall back to that explicitly rather than passing `None` as a chat message's content.

In [ ]:
import sys
sys.path.insert(0, "../src")  # to import data.SYSTEM_PROMPT -- adjust if this notebook moves
from data import SYSTEM_PROMPT as DEFAULT_SYSTEM_PROMPT

SYSTEM_PROMPT = SYSTEM_PROMPT or DEFAULT_SYSTEM_PROMPT

## Load the base model + LoRA adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)  # tokenizer files were saved alongside the adapter

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()
print("loaded.")

## Try it on a Whisper transcript

In [ ]:
def correct(text_whisper: str, max_new_tokens: int = 256) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text_whisper},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
example_whisper_output = "سلام، من نمونه هستم، چطور می‌تونم کمکتون کنم؟"
print(correct(example_whisper_output))

## This run's test-set results (if it trained with `test.dataset_id` set)

In [ ]:
import json

metrics_path = run_dir / "test_eval" / "metrics.json"
if metrics_path.exists():
    print(json.loads(metrics_path.read_text()))
    predictions_path = run_dir / "test_eval" / "predictions.jsonl"
    with open(predictions_path) as f:
        first_rows = [json.loads(next(f)) for _ in range(3)]
    for row in first_rows:
        print(row)
else:
    print("No test_eval/ in this run -- it was trained without test.dataset_id set.")